In [1]:
# | default_exp content_mapper

In [2]:
# | export
from pathlib import Path
from typing import Optional
from seo_rat.index_tracking import fetch_sitemap_urls


In [3]:
# | export
def url_to_relpath(url: str, base_path: str, site_url: str) -> Optional[Path]:
    try:
        url_cleaned = url.removeprefix(site_url).removesuffix(".html")
    except Exception as e:
        print(f"Error processing URL: {url} with site URL: {site_url}. Error: {e}")
        return None
    return Path(base_path) / url_cleaned


In [4]:
# | export
SOURCE_EXTS = (".qmd", ".md", ".ipynb",".mdx",".astro")


def find_source_file(rel_path: Path, exts=SOURCE_EXTS) -> Optional[str]:
    for ext in exts:
        candidate = rel_path.with_suffix(ext)
        if candidate.exists():
            return str(candidate)
    return None


In [5]:
# |export
def url_to_file_path(url: str, base_path: str, site_url: str) -> Optional[str]:
    """Map website URL to local file path"""
    rel_path = url_to_relpath(url, base_path, site_url)
    return find_source_file(rel_path) if rel_path else None


In [6]:
# | export
def build_limax_slug_index(base_path: str) -> dict[str, str]:
    """Build slug → filepath index using Node/limax transliteration (for Astro sites with Arabic filenames)"""
    import subprocess, json
    script = """
    const slugify = require('limax');
    const fs = require('fs');
    const path = require('path');
    const dir = process.env.LIMAX_BASE_PATH;
    const exts = ['.md', '.qmd', '.ipynb', '.mdx', '.astro'];
    const index = {};
    function walk(d) {
        for (const f of fs.readdirSync(d)) {
            const full = path.join(d, f);
            if (fs.statSync(full).isDirectory()) { walk(full); continue; }
            if (exts.some(e => f.endsWith(e))) {
                const name = path.basename(f, path.extname(f));
                index[slugify(name)] = full;
            }
        }
    }
    walk(dir);
    console.log(JSON.stringify(index));
    """
    result = subprocess.run(['node', '-e', script], capture_output=True, text=True, cwd=base_path, env={**__import__('os').environ, 'LIMAX_BASE_PATH': base_path})
    if result.returncode != 0:
        raise RuntimeError(f"limax slug index failed: {result.stderr}")
    return json.loads(result.stdout)


In [7]:
# | export
def build_slug_index(base_path: str) -> dict[str, str]:
    """Build a mapping of slug → filepath by reading frontmatter from all source files"""
    import yaml
    index = {}
    for ext in SOURCE_EXTS:
        for path in Path(base_path).rglob(f"*{ext}"):
            try:
                content = path.read_text(encoding="utf-8")
                if content.startswith("---"):
                    fm = content.split("---")[1]
                    meta = yaml.safe_load(fm)
                    slug = meta.get("slug") if meta else None
                    if slug:
                        index[slug] = str(path)
            except Exception:
                continue
    return index


In [8]:
# | export
from seo_rat.gsc_storage import normalize_url

def map_all_urls_to_files(
    base_path: str, site_url: str, urls: list[str], use_slug_index: bool = False, normalize=False, use_limax: bool = False
) -> dict[str, str]:
    if use_limax:
        slug_index = build_limax_slug_index(base_path)
        return {
            url: slug_index[slug]
            for url in urls
            if (slug := url.removeprefix(site_url).removesuffix("/")) in slug_index
        }
    if use_slug_index:
        slug_index = build_slug_index(base_path)
        return {
            url: slug_index[slug]
            for url in urls
            if (slug := url.removeprefix(site_url).removesuffix("/")) in slug_index
        }
    return {
        normalize_url(url) if normalize else url: path
        for url in urls
        if (path := url_to_file_path(url, base_path, site_url))
    }



{'https://awazly.com/malm-dhanat-aldwadmy': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/معلم دهانات الدوادمي.md',
 'https://awazly.com/malm-dhanat-kfr-alshykh': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/معلم دهانات كفر الشيخ.md',
 'https://awazly.com/ghsyl-knb-baldwadmy': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/غسيل كنب بالدوادمي.md',
 'https://awazly.com/althqb-alafqy': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/الثقب الأفقي.md',
 'https://awazly.com/alkhrsanh-almazwlh': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/الخرسانة المعزولة.md',
 'https://awazly.com/sbak_balmdynh': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/سباك_بالمدينة.md',
 'https://awazly.com/shrkh-tnzyf-mnazl-alkwyt': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/شركة تنظيف منازل الكويت.md',
 'https://awazly.com/shrkh-azl-asth-bmkh': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/شركة عزل اسطح بمكة.md',
 'https://awa

In [ ]:
# | hide
#| eval: false
from dotenv import load_dotenv
import os

load_dotenv()
# BASE_PATH = os.environ["SEO_RAT_BASE_PATH"]
BASE_PATH = os.environ["AWAZLY_PATH"]

urls = fetch_sitemap_urls("https://awazly.com/sitemap.xml")
map_all_urls_to_files(
    base_path=f"{BASE_PATH}",
    site_url="https://awazly.com/",
    urls=urls,
    use_slug_index=True
)
